## 1. SparkSession qurulması 


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import time

spark = (
    SparkSession.builder
    .appName("Lesson20-Fundamentals-Practical")
    .master("spark://spark-master:7077")

    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:/spark-events")

    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "matrix")
    .config("spark.hadoop.fs.s3a.secret.key", "matrix123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

spark.version

'3.5.2'

## 2. Data yaratmaq və yazmaq

In [2]:
df = (
    spark.range(0, 1_000_000)
    .withColumn("category", (F.col("id") % 5).cast("int"))
    .withColumn("amount", (F.rand(seed=42) * 1000).cast("double"))
    .select("category", "amount")
    .repartition(8)
)

base_path = "s3a://matrix/lesson20_base_parquet"
df.write.mode("overwrite").parquet(base_path)
print("write:", base_path)

write: s3a://matrix/lesson20_base_parquet


## 3. Lazy Evaluation nümunəsi

In [3]:
t0 = time.time()

df2 = spark.read.parquet(base_path)          # transformation
df3 = df2.filter(F.col("amount") > 500)      # transformation
df4 = df3.withColumn("tier", F.lit("high"))  # transformation

print(f"Transformations {time.time() - t0:.4f} san")

Transformations 0.5370 san


In [4]:
t0 = time.time()

result_count = df4.count()   

print(f"count() : {result_count}")
print(f"time: {time.time() - t0:.4f} saniyə")


count() : 500360
time: 1.0581 saniyə


## 4. Narrow vs Wide Transformation 



In [5]:
# NARROW: filter —
narrow_result = df2.filter(F.col("category") == 1)
narrow_count = narrow_result.count()
# narrow_result.write.mode("overwrite").parquet("s3a://matrix/narrow_test")
print("Narrow transformation", narrow_count)

Narrow transformation 200000


In [6]:
# WIDE: groupBy + agg
wide_result = (
    df2.repartition(16, "category")
       .groupBy("category")
       .agg(F.sum("amount").alias("total_amount"))
       .orderBy(F.desc("total_amount"))
)
wide_result.show()

+--------+--------------------+
|category|        total_amount|
+--------+--------------------+
|       2| 1.001473232732601E8|
|       3|1.0011397189304553E8|
|       1| 1.000613053894725E8|
|       0|  9.99755698254024E7|
|       4| 9.993867467126794E7|
+--------+--------------------+



In [7]:
distinct_categories = df2.select("category").distinct()
distinct_categories.show()

+--------+
|category|
+--------+
|       1|
|       3|
|       4|
|       2|
|       0|
+--------+



## 5. Physical Plan-a baxış

In [8]:
wide_result.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (7)
+- Sort (6)
   +- Exchange (5)
      +- HashAggregate (4)
         +- HashAggregate (3)
            +- Exchange (2)
               +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [category#13, amount#14]
Batched: true
Location: InMemoryFileIndex [s3a://matrix/lesson20_base_parquet]
ReadSchema: struct<category:int,amount:double>

(2) Exchange
Input [2]: [category#13, amount#14]
Arguments: hashpartitioning(category#13, 16), REPARTITION_BY_NUM, [plan_id=254]

(3) HashAggregate
Input [2]: [category#13, amount#14]
Keys [1]: [category#13]
Functions [1]: [partial_sum(amount#14)]
Aggregate Attributes [1]: [sum#49]
Results [2]: [category#13, sum#50]

(4) HashAggregate
Input [2]: [category#13, sum#50]
Keys [1]: [category#13]
Functions [1]: [sum(amount#14)]
Aggregate Attributes [1]: [sum(amount#14)#39]
Results [2]: [category#13, sum(amount#14)#39 AS total_amount#40]

(5) Exchange
Input [2]: [category#13, total_amount#40]
Arguments: rangepartitioning(

## 6. Job/Stage/Task-ları proqramatik izləmək

In [9]:
tracker = spark.sparkContext.statusTracker()
job_ids = tracker.getJobIdsForGroup()

print(f" SparkSession Job sayı: {len(job_ids)}")
print(f"Job ID-ləri: {sorted(job_ids)}")

if job_ids:
    last_job = tracker.getJobInfo(max(job_ids))
    print(f"\nJob Stage ID: {last_job.stageIds}")

 SparkSession Job sayı: 11
Job ID-ləri: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

Job Stage ID: [I@3c28b486


## 8. Sonlandırma


In [10]:
spark.stop()